
# Porównanie modeli emocji: BiLSTM vs lokalny transformer

Ta wersja jest samowystarczalna:
- jeśli nie ma zapisanych modeli, sama je trenuje,
- nie wymaga pobierania BERT-a,
- zapisuje końcowe metryki i wykres.


In [ ]:

# --- instalacja zależności do Colaba / Pythona 3.12 ---
# Uruchom tę komórkę tylko raz po otwarciu notebooka.
%pip -q install "tensorflow>=2.20,<2.21" "torch>=2.2" "scikit-learn>=1.4" "pandas>=2.2" "numpy>=1.26" "matplotlib>=3.8"


In [ ]:

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["OMP_NUM_THREADS"] = "1"

import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:

CSV_PATH = "emocje_20000.csv"

EMOTION_TEMPLATES = {
    "anger": [
        "Jestem wściekły na tę decyzję.",
        "To mnie doprowadza do szału.",
        "Mam dość tego chaosu i kłamstw.",
        "Ta sytuacja mnie potwornie irytuje.",
        "Nie mogę znieść takiego traktowania.",
    ],
    "disgust": [
        "To jest obrzydliwe i odpychające.",
        "Aż mnie skręca z obrzydzenia.",
        "Nie mogę na to patrzeć.",
        "To budzi we mnie wstręt.",
        "To było wyjątkowo odrażające.",
    ],
    "fear": [
        "Boję się, że wszystko zaraz się zawali.",
        "Czuję silny lęk przed tym, co będzie dalej.",
        "Mam w sobie niepokój i drżenie.",
        "Serce bije mi szybciej ze strachu.",
        "To mnie naprawdę przeraża.",
    ],
    "joy": [
        "Jestem dziś naprawdę szczęśliwy.",
        "Rozpiera mnie radość i energia.",
        "To wspaniała wiadomość, aż się uśmiecham.",
        "Mam świetny humor i dużo optymizmu.",
        "Czuję czystą radość.",
    ],
    "sadness": [
        "Jest mi bardzo smutno po tej wiadomości.",
        "Czuję pustkę i przygnębienie.",
        "Łzy same napływają mi do oczu.",
        "To wszystko budzi we mnie głęboki smutek.",
        "Mam dziś ciężkie serce.",
    ],
    "surprise": [
        "Ale niespodzianka, kompletnie się tego nie spodziewałem.",
        "To było bardzo zaskakujące.",
        "Jestem w szoku po tym zwrocie akcji.",
        "Nie mogę uwierzyć, że to się wydarzyło.",
        "To całkowicie mnie zaskoczyło.",
    ],
}

MODIFIERS = [
    " dzisiaj.", " rano.", " wieczorem.", " po tej rozmowie.",
    " po spotkaniu.", " w pracy.", " w domu.", " i nie wiem co powiedzieć."
]

def ensure_demo_csv(csv_path=CSV_PATH, train_per_label=40, val_per_label=10, test_per_label=10):
    path = Path(csv_path)
    if path.exists():
        print(f"Używam istniejącego pliku: {path}")
        return path

    rows = []
    label_to_id = {label: idx for idx, label in enumerate(EMOTION_TEMPLATES.keys())}

    for label, templates in EMOTION_TEMPLATES.items():
        for split, n in [("train", train_per_label), ("val", val_per_label), ("test", test_per_label)]:
            for _ in range(n):
                text = random.choice(templates) + random.choice(MODIFIERS)
                rows.append({
                    "text": text,
                    "label": label,
                    "label_id": label_to_id[label],
                    "split": split,
                })

    df_demo = pd.DataFrame(rows)
    df_demo.to_csv(path, index=False)
    print(f"Wygenerowano plik demo: {path}")
    return path

ensure_demo_csv()


In [ ]:

df = pd.read_csv(CSV_PATH)
required_cols = {"text", "label", "label_id", "split"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Brakuje kolumn: {missing}")

df = df.dropna(subset=["text", "label", "label_id", "split"]).copy()
df["text"] = df["text"].astype(str)
df["label_id"] = df["label_id"].astype(int)

label_to_id = (
    df[["label", "label_id"]]
    .drop_duplicates()
    .sort_values("label_id")
    .set_index("label")['label_id']
    .to_dict()
)
id_to_label = {int(v): k for k, v in label_to_id.items()}
label_names = [id_to_label[i] for i in sorted(id_to_label.keys())]

print(df.head())
print("\nRozkład splitów:")
print(df["split"].value_counts())
print("\nMapowanie:", label_to_id)


## BiLSTM

In [ ]:

import tensorflow as tf
from tensorflow.keras import layers

COMPARE_OUTPUT_DIR = Path("comparison_output")
COMPARE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TF_MODEL_PATH = Path("model_emocje_tf.keras")
TRANSFORMER_MODEL_PATH = Path("tiny_transformer_output/model_transformer.keras")

MAX_TOKENS = 2000
SEQ_LEN = 24
BATCH_SIZE = 16
EPOCHS_BILSTM = 6

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQ_LEN,
)
vectorizer.adapt(train_df["text"].values)

if TF_MODEL_PATH.exists():
    print(f"Wczytuję istniejący model BiLSTM: {TF_MODEL_PATH}")
    bilstm_model = tf.keras.models.load_model(TF_MODEL_PATH)
else:
    print("Nie znaleziono BiLSTM. Trenuję nowy model...")
    inputs = layers.Input(shape=(1,), dtype=tf.string)
    x = vectorizer(inputs)
    x = layers.Embedding(MAX_TOKENS, 64)(x)
    x = layers.Bidirectional(layers.LSTM(32))(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(len(label_names), activation="softmax")(x)
    bilstm_model = tf.keras.Model(inputs=inputs, outputs=outputs)
    bilstm_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    bilstm_model.fit(
        train_df["text"].values,
        train_df["label_id"].values,
        validation_data=(val_df["text"].values, val_df["label_id"].values),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS_BILSTM,
        verbose=1,
    )
    bilstm_model.save(TF_MODEL_PATH)

bilstm_probs = bilstm_model.predict(test_df["text"].values, verbose=0)
bilstm_pred = np.argmax(bilstm_probs, axis=1)
y_true = test_df["label_id"].values

bilstm_metrics = {
    "accuracy": float(accuracy_score(y_true, bilstm_pred)),
    "f1_macro": float(f1_score(y_true, bilstm_pred, average="macro")),
    "f1_weighted": float(f1_score(y_true, bilstm_pred, average="weighted")),
}

print("Metryki BiLSTM:")
print(bilstm_metrics)


## Lokalny transformer

In [ ]:

@tf.keras.utils.register_keras_serializable()
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.maxlen = maxlen
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

    def get_config(self):
        config = super().get_config()
        config.update({"maxlen": self.maxlen, "vocab_size": self.vocab_size, "embed_dim": self.embed_dim})
        return config

@tf.keras.utils.register_keras_serializable()
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.rate = rate
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([layers.Dense(ff_dim, activation="relu"), layers.Dense(embed_dim)])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

    def get_config(self):
        config = super().get_config()
        config.update({"embed_dim": self.embed_dim, "num_heads": self.num_heads, "ff_dim": self.ff_dim, "rate": self.rate})
        return config

if TRANSFORMER_MODEL_PATH.exists():
    print(f"Wczytuję istniejący model transformera: {TRANSFORMER_MODEL_PATH}")
    transformer_model = tf.keras.models.load_model(TRANSFORMER_MODEL_PATH)
else:
    print("Nie znaleziono modelu transformera. Trenuję go teraz lokalnie...")
    vocab_size = len(vectorizer.get_vocabulary())
    inputs = layers.Input(shape=(1,), dtype=tf.string)
    x = vectorizer(inputs)
    x = TokenAndPositionEmbedding(SEQ_LEN, vocab_size, 32)(x)
    x = TransformerBlock(32, 2, 64)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.1)(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(len(label_names), activation="softmax")(x)
    transformer_model = tf.keras.Model(inputs=inputs, outputs=outputs)
    transformer_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    transformer_model.fit(
        train_df["text"].values,
        train_df["label_id"].values,
        validation_data=(val_df["text"].values, val_df["label_id"].values),
        batch_size=BATCH_SIZE,
        epochs=6,
        verbose=1,
    )
    TRANSFORMER_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    transformer_model.save(TRANSFORMER_MODEL_PATH)

transformer_probs = transformer_model.predict(test_df["text"].values, verbose=0)
transformer_pred = np.argmax(transformer_probs, axis=1)

transformer_metrics = {
    "accuracy": float(accuracy_score(y_true, transformer_pred)),
    "f1_macro": float(f1_score(y_true, transformer_pred, average="macro")),
    "f1_weighted": float(f1_score(y_true, transformer_pred, average="weighted")),
}

print("Metryki transformera:")
print(transformer_metrics)


## Tabela porównawcza

In [ ]:

comparison_df = pd.DataFrame([
    {"model": "BiLSTM (TensorFlow)", **bilstm_metrics, "parametry": int(bilstm_model.count_params())},
    {"model": "Lokalny transformer", **transformer_metrics, "parametry": int(transformer_model.count_params())},
])

comparison_df


In [ ]:

plot_df = comparison_df.set_index("model")[["accuracy", "f1_macro", "f1_weighted"]]
ax = plot_df.plot(kind="bar", figsize=(10, 5))
ax.set_title("Porównanie jakości modeli")
ax.set_ylabel("Wartość metryki")
ax.set_xlabel("Model")
ax.set_ylim(0, 1.0)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

comparison_df.to_csv(COMPARE_OUTPUT_DIR / "comparison_metrics.csv", index=False)
plt.figure(figsize=(10, 5))
ax = plot_df.plot(kind="bar")
plt.tight_layout()
plt.savefig(COMPARE_OUTPUT_DIR / "comparison_plot.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Zapisano wyniki do: {COMPARE_OUTPUT_DIR.resolve()}")


## Trudniejsze przykłady

In [ ]:

challenging_examples = [
    "Niby się cieszę, ale czuję też napięcie przed wynikiem.",
    "To było tak dziwne, że aż mnie odrzuciło.",
    "Nie wiem, czy bardziej jestem zły, czy po prostu rozczarowany.",
    "Serce mi stanęło, gdy zobaczyłem tę wiadomość.",
    "Łzy napłynęły mi do oczu, choć próbowałem się uśmiechać.",
    "Ale numer, kompletnie mnie tym zaskoczyli.",
]

bilstm_example_probs = bilstm_model.predict(np.array(challenging_examples), verbose=0)
bilstm_example_pred = np.argmax(bilstm_example_probs, axis=1)

transformer_example_probs = transformer_model.predict(np.array(challenging_examples), verbose=0)
transformer_example_pred = np.argmax(transformer_example_probs, axis=1)

pd.DataFrame({
    "text": challenging_examples,
    "BiLSTM_pred": [label_names[i] for i in bilstm_example_pred],
    "BiLSTM_conf": [float(np.max(row)) for row in bilstm_example_probs],
    "Transformer_pred": [label_names[i] for i in transformer_example_pred],
    "Transformer_conf": [float(np.max(row)) for row in transformer_example_probs],
})
